[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Validating Requests


## What you will be able to do

Check what a client sends before a route uses it: a JSON body read into a Pydantic model with rules
for its fields, query parameters with limits, and a `422` that names every value that broke a rule.
Decide what a response may contain with a response model, and give a `422` the shape your clients
read.


## The idea

### The problem

The **Sending Data** notebook sent the practice API a latitude of 170, latitudes as text read from a
CSV file, and plans with a field left out, and the practice API refused each one with a `422` that
said why. A route that takes a body as it arrives does none of that. It stores the latitude of 170 as
though a station could stand there, fails with a `500` when a field it reads is missing, and keeps a
misspelled field name without a word, so every client learns about its mistake later, from bad data.

The practice API checks a plan with a function of its own, `plan_problems`, one rule at a time, and
the rules live only in that function: a client cannot read them before it sends a request.

### What request validation is

> **Request validation** checks what a request holds against declared rules before the code that
> uses it runs. In FastAPI, a route parameter whose type is a Pydantic model is the request's
> **body**: FastAPI reads the body as JSON, checks it against the model, and passes the route a model
> object. A body that breaks the model gets `422`, with a `detail` list that holds an entry for every
> problem: where it is, as `loc`, what is wrong, as `msg`, a `type` a program can match, and the
> `input` that broke the rule. `Field` puts rules on a model's fields, such as `ge=-90` and `le=90`
> for a latitude, and `Query` puts the same kind of rule on a query parameter. A **response model**
> checks the other direction: what a route sends is checked against it, and cut down to its fields.

### Why it works that way

- **A rule is declared once, and used three ways.** The model checks every request, hands the route
  values of the declared types, and becomes a JSON Schema in the OpenAPI document, so a client can
  read the rules before it sends anything.
- **Every problem is reported at once.** Pydantic checks every field before it answers, so a body
  with two mistakes gets two entries, and a client can correct both before it tries again.
- **Pydantic converts what it can, unless a field is strict.** The text `"70.05"` becomes the number
  70.05, which suits a query string, where every value is text, and hides a client's bug in a JSON
  body. `strict=True` on a field refuses a value of the wrong JSON type.
- **A field that a model does not declare is dropped, unless the model forbids it.**
  `extra="forbid"` turns a misspelled field name into a `422`, instead of a value that disappears.
- **A response model guards what leaves the server.** FastAPI checks a route's answer against it,
  describes it in the OpenAPI document, and sends only the fields it declares, which FastAPI's
  documentation calls particularly important for security.

### Where you will meet this

FastAPI's documentation shows a response model keeping a password out of a response: the route
takes a model with the password in it, and answers with a model that has none. Stripe's API names a
request with invalid parameters an `invalid_request_error`, and puts the parameter at fault in the
error's `param`, so that a form can show the message beside the right field. GitHub's REST API
answers a request it could not process with `422`, the message `Validation Failed`, and an `errors`
list whose entries carry a `code`, such as `missing_field` for a required parameter left out. The
practice API's `422` for a plan is the same idea in a shape of its own, and this notebook's last
section gives a FastAPI app that shape.

### What this notebook covers

- A JSON body as a Pydantic model, and the `422` for a body that breaks it
- Rules on fields with `Field`, and a rule of your own with `field_validator`
- Strict fields, which refuse numbers sent as text, and `extra="forbid"` for unknown fields
- Rules on query parameters, with `Query`
- The model's rules in the OpenAPI document, where a client can read them
- A response model, which checks and filters what a route sends
- A `422` in a shape of your own, with an exception handler
- Plans checked as the practice API checks them, and refused in its shape
- Five errors, from a model made strict as a whole to a model checked inside the route

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests
from fastapi import FastAPI
from pydantic import BaseModel, Field

import practice_api


class Plan(BaseModel):
    name: str
    latitude: float = Field(ge=-90, le=90)
    longitude: float = Field(ge=-180, le=180)


app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
body = {"name": "Lakselv", "latitude": 170.05}
response = requests.post(f"{app_url}/plans", json=body, timeout=10)
print(response.status_code)
for problem in response.json()["detail"]:
    print(problem["loc"], problem["msg"])
```

```
422
['body', 'latitude'] Input should be less than or equal to 90
['body', 'longitude'] Field required
```

The route has no checks of its own, and it never ran. FastAPI refused the body with both of its
problems, each with where it is and what is wrong.


## Setup

Sixteen imports, the last of them the practice API.

- `requests` sends the bodies, as it did in the **Sending Data** notebook
- `FastAPI` makes an app
- `Query` puts rules on a query parameter
- `RequestValidationError` is what FastAPI raises for a request that breaks its rules, and what a
  handler of your own receives
- `JSONResponse` is the response that handler sends
- `BaseModel` declares a model, as in the **Schemas and Validation** notebook
- `Field` puts rules on a model's fields
- `field_validator` makes a method a rule of your own for a field
- `ConfigDict` sets rules for a whole model, such as `extra="forbid"`
- `Annotated` attaches `Query`'s rules to a parameter's type
- `date` is the type of a plan's `opens`
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, and its `serve` runs the apps in this notebook


In [1]:
import importlib
import sys
import urllib.request
from datetime import date
from pathlib import Path
from typing import Annotated

import requests
from fastapi import FastAPI, Query
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field, field_validator

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### A body as a model

A parameter whose type is a Pydantic model is the request's body. FastAPI reads the body as JSON,
checks it against the model, and passes the route a `Plan`. `status_code=201` in the decorator is the
status the route answers with when it succeeds, `201 Created`, as for the practice API's plans:


In [2]:
class Plan(BaseModel):
    name: str
    latitude: float
    longitude: float
    elevation_m: float | None = None
    opens: date | None = None


app = FastAPI()
plans = []


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    plans.append(plan)
    return {"id": len(plans), **plan.model_dump()}


app_url = practice_api.serve(app)
body = {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "opens": "2027-06-01"}
response = requests.post(f"{app_url}/plans", json=body, timeout=10)

print(response.status_code, response.json())
print(type(plans[0]).__name__, "| opens:", repr(plans[0].opens))


201 {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': None, 'opens': '2027-06-01'}
Plan | opens: datetime.date(2027, 6, 1)


The route received a `Plan`, whose `opens` is a `date` rather than the text the client sent, and
returned a dictionary that FastAPI turned back into JSON, with the date as text again. FastAPI's
documentation sorts a route's parameters three ways: a name in the path is a path parameter, a
parameter of a single type such as `int` or `str` is a query parameter, and a parameter whose type is
a Pydantic model is the body. The route has no code that checks anything, and needs none.

### A body that breaks the model

Three bodies that the model refuses: a latitude that is not a number with the longitude left out, a
date that is not a date, and a list where an object belongs:


In [3]:
bodies = [{"name": "Lakselv", "latitude": "north"},
          {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "opens": "next June"},
          [69.97, 23.27]]

for body in bodies:
    response = requests.post(f"{app_url}/plans", json=body, timeout=10)
    print(response.status_code)
    for problem in response.json()["detail"]:
        print("   ", problem["loc"], problem["type"], "|", problem["msg"], "| input:", problem["input"])
print("plans kept:", len(plans))


422
    ['body', 'latitude'] float_parsing | Input should be a valid number, unable to parse string as a number | input: north
    ['body', 'longitude'] missing | Field required | input: {'name': 'Lakselv', 'latitude': 'north'}
422
    ['body', 'opens'] date_from_datetime_parsing | Input should be a valid date or datetime, input is too short | input: next June
422
    ['body'] model_attributes_type | Input should be a valid dictionary or object to extract fields from | input: [69.97, 23.27]
plans kept: 1


Every problem came back, each with its place in the request, such as `["body", "latitude"]`, its
`type`, a message and the input that broke the rule, and for a missing field that input is the whole
body. The route never ran, so `plans` still holds only the plan from above. This is the `422` that
the **Status Codes** notebook said FastAPI sends, with the reasons that the practice API's `problems`
gave, in FastAPI's shape.

### Rules for fields: Field, and a rule of your own

A type is one rule. `Field` adds others: `ge` and `le` for a number's lowest and highest values, and
`min_length` and `max_length` for text, the kind of rules the **Schemas and Validation** notebook put
on a model. A rule that `Field` cannot say is a method under `@field_validator`, which runs after the
field's type is checked. Raising `ValueError` there reports a problem, and what the method returns
becomes the field's value:


In [4]:
class Plan(BaseModel):
    name: str = Field(min_length=1, max_length=50)
    latitude: float = Field(ge=-90, le=90)
    longitude: float = Field(ge=-180, le=180)
    elevation_m: float | None = None
    opens: date | None = None

    @field_validator("name")
    @classmethod
    def name_is_not_blank(cls, name):
        if not name.strip():
            raise ValueError("must not be blank")
        return name.strip()


app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
for body in [{"name": "Lakselv", "latitude": 170.05, "longitude": 24.97},
             {"name": "   ", "latitude": 69.97, "longitude": 23.27},
             {"name": "  Alta  ", "latitude": 69.97, "longitude": 23.27}]:
    response = requests.post(f"{app_url}/plans", json=body, timeout=10)
    answer = response.json()
    if response.status_code == 422:
        answer = [(problem["loc"][-1], problem["msg"]) for problem in answer["detail"]]
    print(response.status_code, answer)


422 [('latitude', 'Input should be less than or equal to 90')]
422 [('name', 'Value error, must not be blank')]
201 {'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': None, 'opens': None}


Pydantic put `Value error, ` before the validator's message, and gave the problem the `type`
`value_error`, which tells a client the rule is the app's own rather than one of Pydantic's. The
name `  Alta  ` passed, and came back as `Alta`, because the validator returned it stripped. A route
can return the model itself, as this one does, and FastAPI sends its fields as JSON.

### Strict fields, and fields a model does not know

Pydantic converts a value when it can. That suits a query string, where every value is text, and it
hides a mistake in a JSON body, where a number should arrive as a number. `strict=True` on a field
refuses a value of the wrong type, and `extra="forbid"` in the model's `model_config` refuses a field
that the model does not declare, which Pydantic would otherwise drop:


In [5]:
class Plan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=1, max_length=50)
    latitude: float = Field(ge=-90, le=90, strict=True)
    longitude: float = Field(ge=-180, le=180, strict=True)
    elevation_m: float | None = Field(default=None, strict=True)
    opens: date | None = None

    @field_validator("name")
    @classmethod
    def name_is_not_blank(cls, name):
        if not name.strip():
            raise ValueError("must not be blank")
        return name.strip()


app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
for body in [{"name": "Lakselv", "latitude": "70.05", "longitude": "24.97"},          # a row read from a CSV file
             {"name": "Hasvik", "latitude": 70.49, "longitude": 22.14, "elevation": 20},
             {"name": "Alta", "latitude": 70, "longitude": 23.27, "opens": "2027-06-01"}]:
    response = requests.post(f"{app_url}/plans", json=body, timeout=10)
    answer = response.json()
    if response.status_code == 422:
        answer = [(problem["loc"][-1], problem["msg"]) for problem in answer["detail"]]
    print(response.status_code, answer)


422 [('latitude', 'Input should be a valid number'), ('longitude', 'Input should be a valid number')]
422 [('elevation', 'Extra inputs are not permitted')]
201 {'name': 'Alta', 'latitude': 70.0, 'longitude': 23.27, 'elevation_m': None, 'opens': '2027-06-01'}


The numbers read as text were refused, as the practice API refused them in the **Sending Data**
notebook, and so was `elevation`, which the model does not declare. The last plan shows what strict
fields still take: a whole number where a float is declared, since JSON has one kind of number and
does not tell `70` from `70.0`, and a date sent as text, since `opens` is not strict. Common errors
shows why strictness belongs on the fields that need it, and not on the whole model.

### Rules for query parameters

A query parameter takes rules the same way, from `Query`, attached to its type with `Annotated`,
which FastAPI's documentation recommends. The **Your First API Server** notebook's `limit` was only
an `int`, so a limit of -3 passed, and its slice returned all but three of the readings. With `ge=1`
and `le=100` that cannot happen. This app takes plans too, for the next section:


In [6]:
app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


@app.get("/readings")
def readings(station: str, limit: Annotated[int, Query(ge=1, le=100)] = 3):
    """The latest readings from one station, newest first."""
    found = [reading for reading in practice_api.READINGS if reading["station"] == station]
    return found[::-1][:limit]


app_url = practice_api.serve(app)
for query in ["station=oslo&limit=-3", "station=oslo&limit=101", "station=oslo&limit=2"]:
    response = requests.get(f"{app_url}/readings?{query}", timeout=10)
    answer = response.json()
    shown = answer["detail"][0]["msg"] if response.status_code == 422 else f"{len(answer)} readings"
    print(response.status_code, query, "|", shown)


422 station=oslo&limit=-3 | Input should be greater than or equal to 1
422 station=oslo&limit=101 | Input should be less than or equal to 100
200 station=oslo&limit=2 | 2 readings


`Path`, also from `fastapi`, puts the same kinds of rule on a path parameter.

### The rules, in the OpenAPI document

The model's rules are part of the app's OpenAPI document, as the JSON Schema that the **Schemas and
Validation** notebook generated from a model, so a client can read them before it sends a request:


In [7]:
document = requests.get(f"{app_url}/openapi.json", timeout=10).json()

print("request body:", document["paths"]["/plans"]["post"]["requestBody"]["content"]["application/json"]["schema"])
schema = document["components"]["schemas"]["Plan"]
print("required:", schema["required"], "| additionalProperties:", schema["additionalProperties"])
for name, rules in schema["properties"].items():
    print("   ", name, {key: value for key, value in rules.items() if key != "title"})
limit = next(parameter for parameter in document["paths"]["/readings"]["get"]["parameters"] if parameter["name"] == "limit")
print("limit:", limit["schema"])


request body: {'$ref': '#/components/schemas/Plan'}
required: ['name', 'latitude', 'longitude'] | additionalProperties: False
    name {'type': 'string', 'maxLength': 50, 'minLength': 1}
    latitude {'type': 'number', 'maximum': 90.0, 'minimum': -90.0}
    longitude {'type': 'number', 'maximum': 180.0, 'minimum': -180.0}
    elevation_m {'anyOf': [{'type': 'number'}, {'type': 'null'}]}
    opens {'anyOf': [{'type': 'string', 'format': 'date'}, {'type': 'null'}]}
limit: {'type': 'integer', 'maximum': 100, 'minimum': 1, 'default': 3, 'title': 'Limit'}


The request body is the `Plan` schema: three required fields, no others allowed, and every `Field`
rule as `minimum`, `maximum`, `minLength` or `maxLength`, with `limit`'s rules in its own schema.
The validator's rule is not there: a rule written in Python has no JSON Schema form, so a client
learns of it only from a `422`. Swagger UI shows the same schema under the route, beside an example
body built from it.

### A response model

A route's answer can be checked too. `response_model` names the model the answer must fit: FastAPI
checks the answer against it, and sends only the fields it declares, so a value kept for the
server's own use stays on the server. Here every plan records who created it, and the response
leaves that out. `response_model_exclude_none=True` also leaves out a field whose value is `None`,
as the practice API does:


In [8]:
class PlanOut(BaseModel):
    id: int
    name: str
    latitude: float
    longitude: float
    elevation_m: float | None = None
    opens: date | None = None


app = FastAPI()
plans = []


@app.post("/plans", status_code=201, response_model=PlanOut, response_model_exclude_none=True)
def create_plan(plan: Plan):
    record = {"id": len(plans) + 1, **plan.model_dump(), "created_by": "planning office"}
    plans.append(record)
    return record


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/plans", json={"name": "Alta", "latitude": 69.97, "longitude": 23.27}, timeout=10)
print(response.status_code, response.json())
print("kept on the server:", plans[0])

document = requests.get(f"{app_url}/openapi.json", timeout=10).json()
print("201's schema:", document["paths"]["/plans"]["post"]["responses"]["201"]["content"]["application/json"]["schema"])


201 {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27}
kept on the server: {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': None, 'opens': None, 'created_by': 'planning office'}
201's schema: {'$ref': '#/components/schemas/PlanOut'}


The response held four fields, and the server kept all seven. FastAPI's documentation shows the same
filter keeping a password out of a response. The response model checks as well as filters: an answer
missing a field it requires is not sent at all, and the client gets `500`, since the fault is the
server's.

### A 422 in a shape of your own

FastAPI's `422` has the `detail` shape, and a client written for the practice API reads
`{"error": ..., "problems": [...]}`, as the **A Real Client** notebook's `InvalidRequestError` does.
For a request that breaks the rules, FastAPI raises `RequestValidationError`, and a function under
`@app.exception_handler(RequestValidationError)` becomes the one that answers it, from the same
entries as `detail`, in its `errors()`. A handler belongs in the cell that makes the app, since an
app settles its handlers when it answers its first request:


In [9]:
app = FastAPI()


@app.exception_handler(RequestValidationError)
def plan_problems(request, error):
    """Answer a request that broke the rules in the practice API's shape."""
    problems = [{"field": ".".join(str(part) for part in problem["loc"][1:]) or None, "problem": problem["msg"]}
                for problem in error.errors()]
    return JSONResponse(status_code=422, content={"error": "the plan has problems", "problems": problems})


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
for body in [{"name": "Lakselv", "latitude": 170.05}, [69.97, 23.27]]:
    response = requests.post(f"{app_url}/plans", json=body, timeout=10)
    print(response.status_code, response.json())


422 {'error': 'the plan has problems', 'problems': [{'field': 'latitude', 'problem': 'Input should be less than or equal to 90'}, {'field': 'longitude', 'problem': 'Field required'}]}
422 {'error': 'the plan has problems', 'problems': [{'field': None, 'problem': 'Input should be a valid dictionary or object to extract fields from'}]}


Each `loc` without its first part, `body`, is the field's name, and a problem with the body as a
whole has no field, `None`, as in the practice API's answer to a body that is not an object. The
handler answers for every route in the app, so an app with routes for more than plans would need a
message that fits any request, not `the plan has problems`.

### Plans checked as the practice API checks them

The pieces of this notebook, in one app: a strict `Plan` with rules for every field, a rule of its
own for the name and no fields it does not declare, a response model that keeps the planner's name
on the server, and a `422` in the practice API's shape. The same bodies go to the practice API's
`/network/plans` and to the app's `/plans`, and each line compares the two answers:


In [10]:
app = FastAPI(title="Planned stations")
plans = []


@app.exception_handler(RequestValidationError)
def plan_problems(request, error):
    """Answer a request that broke the rules in the practice API's shape."""
    problems = [{"field": ".".join(str(part) for part in problem["loc"][1:]) or None, "problem": problem["msg"]}
                for problem in error.errors()]
    return JSONResponse(status_code=422, content={"error": "the plan has problems", "problems": problems})


@app.post("/plans", status_code=201, response_model=PlanOut, response_model_exclude_none=True)
def create_plan(plan: Plan):
    """A new plan, checked against every rule of Plan."""
    record = {"id": len(plans) + 1, **plan.model_dump(), "created_by": "planning office"}
    plans.append(record)
    return record


app_url = practice_api.serve(app)


def outcome(response):
    """The status code, with the fields a 422 names, or the name of the plan a 201 made."""
    answer = response.json()
    if response.status_code == 422:
        return response.status_code, [problem["field"] for problem in answer["problems"]]
    return response.status_code, answer["name"]


bodies = {
    "a plan": {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "opens": "2027-06-01"},
    "a latitude of 170, no longitude": {"name": "Lakselv", "latitude": 170.05},
    "numbers read as text": {"name": "Lakselv", "latitude": "70.05", "longitude": "24.97"},
    "a blank name": {"name": "   ", "latitude": 69.97, "longitude": 23.27},
    "a date that is not one": {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "opens": "next June"},
    "a list, not an object": [69.97, 23.27],
    "a misspelled field": {"name": "Hasvik", "latitude": 70.49, "longitude": 22.14, "elevation": 20},
}
for label, body in bodies.items():
    practice = requests.post(f"{BASE}/network/plans", json=body, timeout=10)
    ours = requests.post(f"{app_url}/plans", json=body, timeout=10)
    print(f"{label:<32} practice API {outcome(practice)} | this app {outcome(ours)}")


a plan                           practice API (201, 'Alta') | this app (201, 'Alta')
a latitude of 170, no longitude  practice API (422, ['latitude', 'longitude']) | this app (422, ['latitude', 'longitude'])
numbers read as text             practice API (422, ['latitude', 'longitude']) | this app (422, ['latitude', 'longitude'])
a blank name                     practice API (422, ['name']) | this app (422, ['name'])
a date that is not one           practice API (422, ['opens']) | this app (422, ['opens'])
a list, not an object            practice API (422, [None]) | this app (422, [None])
a misspelled field               practice API (201, 'Hasvik') | this app (422, ['elevation'])


### Where each part came from

| In the app | What it relies on | The section that showed it |
|---|---|---|
| `plan: Plan` | a parameter typed as a model, read from the JSON body | A body as a model |
| `status_code=201` | the status a route sends when it succeeds | A body as a model |
| `Field(ge=-90, le=90, ...)` | rules on a field | Rules for fields: Field, and a rule of your own |
| `name_is_not_blank` | a rule of your own, with `field_validator` | Rules for fields: Field, and a rule of your own |
| `strict=True` and `extra="forbid"` | numbers only as numbers, and no fields the model does not declare | Strict fields, and fields a model does not know |
| `response_model=PlanOut` | an answer checked, and cut down to the model's fields | A response model |
| `@app.exception_handler(RequestValidationError)` | a `422` in the practice API's shape | A 422 in a shape of your own |

Six bodies of the seven got the same status code and the same fields from both servers, from rules
written as a model rather than as a function. The seventh is where this app is stricter: the practice
API ignores a field it does not know, and made a plan for Hasvik with no elevation, while
`extra="forbid"` refused it. The messages differ, `Input should be less than or equal to 90` against
`must be a number from -90 to 90`, because this app's come from Pydantic, and a client that reads
`field` handles both.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/16-validating-requests-solutions.ipynb).

**1.** Make an app with a route, `POST /readings`, whose body is a model `Reading` with a `station`
and a `temperature_c` that is a float, and which answers `201` with the reading. Send
`{"station": "oslo", "temperature_c": -4.2}`, and print the status code and the body.


In [11]:
# your code here


**2.** Give `temperature_c` the rule that it is from -90 to 60. Send a reading of 75, and print each
problem's `loc` and `msg`.


In [12]:
# your code here


**3.** Make `temperature_c` strict, and forbid fields that `Reading` does not declare. Send
`{"station": "oslo", "temperature_c": "-4.2", "unit": "C"}`, and print each problem's `loc` and
`msg`.


In [13]:
# your code here


**4.** Add a validator to `station` that turns it to lower case and refuses an id that is not in
`practice_api.STATIONS`. Send the station as `Oslo`, then as `narvik`, and print both answers.


In [14]:
# your code here


**5.** Make an app with a route, `GET /readings`, whose query parameter `hours` must be from 1 to 72,
with a default of 24. Request `hours=0`, and print the problem's `msg`.


In [15]:
# your code here


**6.** Make an app with the route from task 4 and an exception handler that answers a `422` as
`{"error": "the reading has problems", "problems": [...]}`, with a `field` and a `problem` for each.
Send the body from task 3 again, and print the answer.


In [16]:
# your code here


## Common errors

### 422, Input should be a valid date: a model made strict as a whole


In [17]:
class StrictPlan(BaseModel):
    model_config = ConfigDict(strict=True)

    name: str
    latitude: float
    longitude: float
    opens: date | None = None


app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: StrictPlan):
    return plan


app_url = practice_api.serve(app)
body = {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "opens": "2027-06-01"}
response = requests.post(f"{app_url}/plans", json=body, timeout=10)
print(response.status_code, response.json())


422 {'detail': [{'type': 'date_type', 'loc': ['body', 'opens'], 'msg': 'Input should be a valid date', 'input': '2027-06-01'}]}


`strict=True` for the whole model made `opens` strict as well. FastAPI reads a body into Python
values before it checks them, so `opens` arrived as the text `"2027-06-01"`, and a strict `date`
field takes only a `date` object, which is what the **Schemas and Validation** notebook met when it
checked `response.json()` in strict mode. JSON has no type for dates, so a date always arrives as
text. Put `strict=True` on the fields that should refuse text, as `Plan` does:


In [18]:
app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/plans", json=body, timeout=10)
print(response.status_code, response.json())


201 {'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': None, 'opens': '2027-06-01'}


### No error, and a number read as text: a field that is not strict


In [19]:
class LaxPlan(BaseModel):
    name: str
    latitude: float = Field(ge=-90, le=90)
    longitude: float = Field(ge=-180, le=180)
    elevation_m: float | None = None


app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: LaxPlan):
    return plan


app_url = practice_api.serve(app)
row = {"name": "Lakselv", "latitude": "70.05", "longitude": "24.97"}          # a row read from a CSV file
response = requests.post(f"{app_url}/plans", json=row, timeout=10)
print(response.status_code, response.json())


201 {'name': 'Lakselv', 'latitude': 70.05, 'longitude': 24.97, 'elevation_m': None}


Pydantic turned the text into numbers, so the plan was made, and the client never learns that it
sends its coordinates as text, which the practice API refused in the **Sending Data** notebook. The
next API that client sends to may not convert them. Make the fields strict, as `Plan`'s are:


In [20]:
app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/plans", json=row, timeout=10)
print(response.status_code, [(problem["loc"][-1], problem["msg"]) for problem in response.json()["detail"]])


422 [('latitude', 'Input should be a valid number'), ('longitude', 'Input should be a valid number')]


### No error, and an elevation gone: a misspelled field the model ignores


In [21]:
app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: LaxPlan):
    return plan


app_url = practice_api.serve(app)
body = {"name": "Hasvik", "latitude": 70.49, "longitude": 22.14, "elevation": 20}
response = requests.post(f"{app_url}/plans", json=body, timeout=10)
print(response.status_code, response.json())


201 {'name': 'Hasvik', 'latitude': 70.49, 'longitude': 22.14, 'elevation_m': None}


`LaxPlan` has no field called `elevation`, so Pydantic dropped it, and `elevation_m` took its
default, `None`. Nothing failed, and the plan lost its elevation, as it does in the practice API
with the same body. `extra="forbid"` refuses a field the model does not declare, as `Plan`'s
`model_config` does:


In [22]:
app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/plans", json=body, timeout=10)
print(response.status_code, response.json())


422 {'detail': [{'type': 'extra_forbidden', 'loc': ['body', 'elevation'], 'msg': 'Extra inputs are not permitted', 'input': 20}]}


### HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/plans


In [23]:
app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(body: dict):
    plan = Plan(**body)                   # checked inside the route
    return plan


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/plans", json={"name": "Lakselv", "latitude": 170.05}, timeout=10)
response.raise_for_status()


HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/plans

`Plan(**body)` raised Pydantic's `ValidationError` inside the route, where it is an exception like
any other, so the client got a `500` and no reason. FastAPI answers `422` for the rules it checks
before the route runs, and it checks a parameter only against the type the parameter declares, here
a plain `dict`. Declare the parameter as the model:


In [24]:
app = FastAPI()


@app.post("/plans", status_code=201)
def create_plan(plan: Plan):
    return plan


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/plans", json={"name": "Lakselv", "latitude": 170.05}, timeout=10)
print(response.status_code, [(problem["loc"][-1], problem["msg"]) for problem in response.json()["detail"]])


422 [('latitude', 'Input should be less than or equal to 90'), ('longitude', 'Field required')]


### 422, Input should be a valid dictionary or object to extract fields from: a body sent as a form


In [25]:
plan = {"name": "Alta", "latitude": 69.97, "longitude": 23.27}
response = requests.post(f"{app_url}/plans", plan, timeout=10)
print(response.status_code, response.json())


422 {'detail': [{'type': 'model_attributes_type', 'loc': ['body'], 'msg': 'Input should be a valid dictionary or object to extract fields from', 'input': 'name=Alta&latitude=69.97&longitude=23.27'}]}


The dictionary went in the position of `data`, not `json`, as in the **Sending Data** notebook's
Common errors, so requests sent it as a form, and FastAPI reads a model only from a JSON body: it saw
the text `name=Alta&latitude=69.97&longitude=23.27`, not an object. The practice API answers the same
mistake with `415`, and FastAPI with this `422`. Send the body with `json=`:


In [26]:
response = requests.post(f"{app_url}/plans", json=plan, timeout=10)
print(response.status_code, response.json())


201 {'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': None, 'opens': None}


## Recap

- A route parameter typed as a Pydantic model is the request's JSON body, checked before the route
  runs.
- A body that breaks the model gets `422`, with a `detail` entry for every problem: its `loc`, `msg`,
  `type` and `input`.
- `Field` sets rules such as `ge`, `le` and `max_length`, and a method under `@field_validator`
  adds a rule of your own, reported after `Value error, `.
- `strict=True` on a field refuses a value of the wrong JSON type, and `extra="forbid"` refuses a
  field the model does not declare. Leave dates out of strictness, since JSON sends them as text.
- `Annotated[int, Query(ge=1, le=100)]` sets rules on a query parameter, and every rule except a
  validator's appears in the OpenAPI document.
- `response_model` checks what a route sends and keeps it to the model's fields, and an exception
  handler for `RequestValidationError` gives a `422` a shape of your own.


## What is next

The **A Complete API** notebook. This notebook's apps could only create a plan. That notebook
rebuilds the practice API's plans in FastAPI with every method, to read, replace, change and remove a
plan, and with the status code a server chooses for each, from `201` with a `Location` to `204` with
no body.


---

&#8592; **Previous:** [Your First API Server](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/15-your-first-api-server.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
